# F0 · The big picture: text in, next character out

**Tiny Transformer, Unit 1, episode F0.** Our tiny transformer, run as a closed box. You give it a line of text, and it
returns a probability for each of the 65 characters that could come next. Here you run it yourself on the course's line from
*The Tempest*: turn its characters into ids, read the 32-character window, get the 65 probabilities, pick one character
(greedy or sampled), and write a passage by repeating that.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/F0-whole-machine.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook F0-whole-machine.ipynb`.

**Computed here, and recorded.** Every number below is computed in this notebook by the trained model. Its 9,417 learned numbers
are the course's recorded run (`prompter-r1`), loaded from `prompter.json`. The asserts compare what you compute with what the
course recorded: the top 5 after the line, the greedy text and the seed-3 sample. So the page and this notebook cannot disagree.

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, hashlib, json, os, urllib.request
import numpy as np
import torch

torch.set_num_threads(1)     # a model this small runs fastest on one CPU thread
print(torch.__version__)

2.10.0+cpu


**Before section 1, load the model.** Three cells: the text it learned from (Tiny Shakespeare, pinned by its sha256), the model's
code (`model.py`, copied in whole from the course: the same file that trained it), and its trained numbers. In that file the
model's class is named `Prompter`; here it is simply *the model*.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"Tiny Shakespeare: {len(text):,} characters")

Tiny Shakespeare: 1,115,394 characters


In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
tok = Tokenizer(text)                                   # the vocabulary (F0.2)
cfg = Config(vocab_size=len(tok.chars))
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the model's trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy())
         for t in exp['weights']['tensors']}
model = Prompter(cfg)
model.load_state_dict(state)
model.eval()
print(f"run {exp['run']}, step {exp['checkpoint_step']:,}: {count_params(model):,} learned numbers")
assert count_params(model) == 9417 and exp['checkpoint_step'] == 98000

run prompter-r1, step 98,000: 9,417 learned numbers


And the episode's three teaching functions (the course's `code/whole_machine.py`, unchanged): `next_char_probs` runs the model
once, and `write_greedy` and `write_sampled` write with it. F0.3 to F0.6 read them line by line. Both files keep their line
numbers, as the page numbers them.

In [6]:
"""F0 · The big picture: our tiny transformer as a closed box (Tiny Transformer, Unit 1; ADR 0011).

Three teaching functions say what the whole machine does, on top of training/prompter/model.py: one forward pass turns the
last 32 character ids into 65 probabilities for the next character; greedy decoding writes by always taking the most probable;
sampling writes by drawing each next character by its probability, with a seed so the draws can be repeated. Each new
character costs one more forward pass, and nothing inside the model changes while it writes. The episode's notebook
(notebooks/F0-whole-machine.ipynb) carries this file unchanged. test_whole_machine.py checks it against model.py's own forward
pass and generate(), on the recorded weights (run prompter-r1), and against the recorded numbers (prompter-traces.json,
prompter-samples.json, prompter-stages.json).
"""
import torch


@torch.no_grad()
def next_char_probs(model, ids):
    """One forward pass: character ids in (only the last 32 are read), 65 probabilities for the next character out."""
    ids = torch.as_tensor(ids)[-model.cfg.block_size:]                 # the window: the last 32 ids              [32]
    T = len(ids)
    x = model.tok_emb(ids) + model.pos_emb(torch.arange(T))            # a token row + a position row per place   [32, 28]
    x = model.blocks(x.unsqueeze(0)).squeeze(0)                        # the block: attention, then feed-forward  [32, 28]
    logits = model.ln_f(x) @ model.tok_emb.weight.T + model.head_bias  # a score for each of 65 characters        [32, 65]
    return torch.softmax(logits[-1], dim=-1)                           # the last row's scores → probabilities    [65]


def write_greedy(model, ids, n):
    """Greedy decoding: n times, take the most probable next character, append it, and run again. Returns the n new ids."""
    text = [int(i) for i in ids]                                       # the text so far, as a list of ids        [T]
    for _ in range(n):                                                 # one forward pass per new character
        p = next_char_probs(model, text)                               # [65]
        text.append(int(p.argmax()))                                   # the most probable id, appended           [T + 1]
    return text[len(ids):]


def write_sampled(model, ids, n, seed):
    """Sampling: n times, draw the next character at random, each as likely as its probability. The seed fixes the draws."""
    g = torch.Generator().manual_seed(seed)                            # the same seed → the same draws
    text = [int(i) for i in ids]                                       # the text so far, as a list of ids        [T]
    for _ in range(n):                                                 # one forward pass per new character
        p = next_char_probs(model, text)                               # [65]
        text.append(int(torch.multinomial(p, 1, generator=g)))         # one draw, by probability, appended       [T + 1]
    return text[len(ids):]

# Section 1 · Text in

What goes into the model, and its one job: the job itself, how characters become ids, and how much text it reads at once. Each
chapter below follows its page: Picture it, Watch it on our line, then Read and run the code.

## F0.1 · The task: predict the next character

*What job does the model do?*

**Predict first** (the page asks it too, before its clip): the model's list after *“…and my p”* is spread, like the list after
`th`. Its 9,417 learned numbers make it. How did training set them: changed them bit by bit, to predict better; counted which
character follows which; or picked them at random, once?

### F0.1.1 · Picture it

Play the model's game: guess the character that comes next. After `q` it is almost always `u`; after `th` it could be `e`, `a`,
`i` or many others. So one guess is often wrong. A better answer gives each character a number: how likely it comes next.
Those numbers are probabilities, and a list of them, one per character, is all the model returns. (The page's numbers here
are made up to show the idea.)

### F0.1.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Treat the model as a closed box of 9,417 learned numbers, its **parameters**. Training set them, by changing them to
predict better; while it runs they stay fixed. Text goes in: our line, 32 characters of Caliban's speech in *The Tempest*,
from the text held back for testing, which the model never trained on (M1 shows where). Out comes one probability for each of the
65 characters: `r` gets 0.340, `a` 0.210 and `l` 0.133. In the play the next character is
`r`, as in *profit*. That is the whole job: predict the next character.

### F0.1.3 · Read and run the code

The whole job is one function in `code/whole_machine.py` (copied in above), `next_char_probs`: it takes the ids of the text so
far, runs the model once, and returns 65 probabilities. Run the cell on our line: the 5 most probable, each checked against the
recorded run.

In [7]:
LINE = 'You taught me language; and my p'
at = text.find(LINE)
print(repr(text[at:at + 42]))                          # the line in the play
p = next_char_probs(model, tok.encode(LINE))           # one run of the model: F0.4 opens it
top = p.topk(5)
for q, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{tok.chars[i]!r:5} {q:.4f}")
RECORDED = [('r', 0.3403), ('a', 0.2101), ('l', 0.1328), ('o', 0.1061), ('e', 0.1044)]
assert [tok.chars[i] for i in top.indices.tolist()] == [c for c, _ in RECORDED]
assert all(abs(q - r) < 2e-4 for q, (_, r) in zip(top.values.tolist(), RECORDED))
assert tok.chars[p.argmax()] == text[at + len(LINE)] == 'r'       # its most probable is the play's own next character
assert at >= int(0.9 * len(text))                                  # the line is in the held-back part (M1)

"You taught me language; and my profit on't"
'r'   0.3403
'a'   0.2101
'l'   0.1328
'o'   0.1061
'e'   0.1044


### ✋ Your turn

Change the line. Give the model `"O Romeo, Romeo! wherefore art th"` (32 characters of *Romeo and Juliet*) and put
its most probable next character in `answer`. The play goes on *thou*. The model may well disagree: it is a tiny model, and
its most probable is not always the play's own next character.

In [8]:
answer = None   # TODO: tok.chars[next_char_probs(model, tok.encode("O Romeo, Romeo! wherefore art th")).argmax()]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[next_char_probs(model, tok.encode("O Romeo, Romeo! wherefore art th")).argmax()], 'run next_char_probs on the new line, then take the character with the largest probability'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [9]:
#@title Show a solution { display-mode: "form" }
p_romeo = next_char_probs(model, tok.encode("O Romeo, Romeo! wherefore art th"))
answer = tok.chars[p_romeo.argmax()]
print(f"most probable {answer!r} {p_romeo.max().item():.3f}; the play's o gets {p_romeo[tok.stoi['o']].item():.3f}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[next_char_probs(model, tok.encode("O Romeo, Romeo! wherefore art th")).argmax()], 'run next_char_probs on the new line, then take the character with the largest probability'
    print("✓ right:", answer)

most probable 'e' 0.565; the play's o gets 0.090
✓ right: e


## F0.2 · The vocabulary: 65 characters, one id each

*Which characters does it know, and how does text become numbers?*

**Predict first** (the page asks it too, before its clip): Tiny Shakespeare uses 65 different characters. Are capital `R` and
small `r` one of them, or two?

### F0.2.1 · Picture it

Inside, the model does arithmetic: it multiplies and adds numbers, and a letter cannot be multiplied. So each character gets a
number from a fixed list: a code book. Take 5 characters: a space, `a`, `b`, `c` and `d`, numbered 0 to 4; then `cab`
becomes 3, 1, 2. The numbers are names, not amounts: `c` is not "more" than `a`. (Where the picture breaks: the model never
multiplies that number itself; M2 shows what it does with it.)

### F0.2.2 · Watch it on our line

*(The page's recorded clip, in words.)*

For this model a **token** is one character. Its **vocabulary** is every distinct character in Tiny Shakespeare, sorted: 65 of
them, capital and small letters, a space, a new line, the digit 3 and 10 marks. A character's **id** is its place in that list:
capital `Y` is 37, small `o` is 53 and a space is 1. **Tokenization** swaps each character of the line for its id, so the line
becomes 32 numbers: a tensor of shape `[32]`. From here on, the model only ever works with numbers.

### F0.2.3 · Read and run the code

The page lights `class Tokenizer` in `model.py` (copied in above): `chars` keeps each character of the text once, sorted;
`stoi` gives each its place, its id; `encode` swaps each character for its id and `decode` turns ids back into text. Run the
two cells: the vocabulary, then our line as 32 ids and back.

In [10]:
print(len(tok.chars), 'characters in the vocabulary, as id and character:')
for row in range(13):
    print('   '.join(f"{i:2d} {tok.chars[i]!r:5}" for i in range(5 * row, 5 * row + 5)))
marks = [c for c in tok.chars if not (c.isalpha() or c in ' \n3')]
print(sum(c.isupper() for c in tok.chars), 'capitals,', sum(c.islower() for c in tok.chars), 'small letters, a new line, a space, the digit 3 and',
      len(marks), 'marks:', ' '.join(marks))
assert len(tok.chars) == 65 and len(marks) == 10

65 characters in the vocabulary, as id and character:
 0 '\n'     1 ' '      2 '!'      3 '$'      4 '&'  
 5 "'"      6 ','      7 '-'      8 '.'      9 '3'  
10 ':'     11 ';'     12 '?'     13 'A'     14 'B'  
15 'C'     16 'D'     17 'E'     18 'F'     19 'G'  
20 'H'     21 'I'     22 'J'     23 'K'     24 'L'  
25 'M'     26 'N'     27 'O'     28 'P'     29 'Q'  
30 'R'     31 'S'     32 'T'     33 'U'     34 'V'  
35 'W'     36 'X'     37 'Y'     38 'Z'     39 'a'  
40 'b'     41 'c'     42 'd'     43 'e'     44 'f'  
45 'g'     46 'h'     47 'i'     48 'j'     49 'k'  
50 'l'     51 'm'     52 'n'     53 'o'     54 'p'  
55 'q'     56 'r'     57 's'     58 't'     59 'u'  
60 'v'     61 'w'     62 'x'     63 'y'     64 'z'  
26 capitals, 26 small letters, a new line, a space, the digit 3 and 10 marks: ! $ & ' , - . : ; ?


In [11]:
ids = torch.tensor(tok.encode(LINE))                   # tokenization: one id per character
print(ids.tolist())
print('shape', tuple(ids.shape))
print('Y is', tok.stoi['Y'], '· o is', tok.stoi['o'], '· a space is', tok.stoi[' '])
print('decoded back:', repr(tok.decode(ids)))
assert tuple(ids.shape) == (32,) and ids.tolist() == [37, 53, 59, 1, 58, 39, 59, 45, 46, 58, 1, 51, 43, 1, 50, 39, 52, 45, 59, 39, 45, 43, 11, 1, 39, 52, 42, 1, 51, 63, 1, 54] and tok.decode(ids) == LINE
assert (tok.stoi['Y'], tok.stoi['o'], tok.stoi[' ']) == (37, 53, 1)

[37, 53, 59, 1, 58, 39, 59, 45, 46, 58, 1, 51, 43, 1, 50, 39, 52, 45, 59, 39, 45, 43, 11, 1, 39, 52, 42, 1, 51, 63, 1, 54]
shape (32,)
Y is 37 · o is 53 · a space is 1


decoded back: 'You taught me language; and my p'


### ✋ Your turn

Tokenize the word `"profit"`: put its list of ids in `answer`. Then turn the ids back into text with `tok.decode`.

In [12]:
answer = None   # TODO: tok.encode(...)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.encode("profit"), 'tok.encode("profit"): six characters, six ids'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [13]:
#@title Show a solution { display-mode: "form" }
answer = tok.encode("profit")
print(answer, repr(tok.decode(answer)))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.encode("profit"), 'tok.encode("profit"): six characters, six ids'
    print("✓ right:", answer)

[54, 56, 53, 44, 47, 58] 'profit'
✓ right: [54, 56, 53, 44, 47, 58]


## F0.3 · The window: the last 32 characters

*How much text does it read at once?*

**Predict first** (the page asks it too, before its clip): a character 40 places back, outside the window: how much does it
change the model's next prediction?

### F0.3.1 · Picture it

To guess the next character, how far back should you look? Shown only the last letter, `m`, many letters could follow.
Shown `on the m`, fewer could; shown `the cat sat on the m`, your best guess is `a`, for *mat*. More text before narrows the
guess, but each character further back usually adds less. A model must stop somewhere: its window.

### F0.3.2 · Watch it on our line

*(The page's recorded clip, in words.)*

The 32 ids go in together, all at once. 32 is the most the model can read at once: its **context**, and our line fills it
exactly (it has a learned row for each place 0 to 31, and none for a 33rd). The whole line is longer: to predict past
*profit*, the window holds the last 32, and the first 5 characters have fallen out. Anything outside the window does not exist
for the model; inside it, every character can matter.

### F0.3.3 · Read and run the code

The window is the first line of `next_char_probs`: its slice keeps the last `block_size` ids, and `block_size` is 32.
Everything before them is cut off before the model sees it. Run the cell on the line up to *profit*: with or without the 5
dropped characters, the 65 probabilities are exactly the same.

In [14]:
FULL = text[at:at + 42]                                # the whole line, as the play has it
upto = FULL[:FULL.index('profit') + len('profit')]     # up to "profit"
window = upto[-cfg.block_size:]                        # what the model reads: the last 32
print(len(upto), 'characters up to "profit"; the window:', repr(window), '; dropped out:', repr(upto[:-cfg.block_size]))
print('one learned position row per place:', tuple(model.pos_emb.weight.shape))
same = torch.equal(next_char_probs(model, tok.encode(upto)), next_char_probs(model, tok.encode(window)))
print('the same 65 probabilities with or without the dropped characters:', same)
assert FULL == "You taught me language; and my profit on't" and len(upto) == 37
assert window == 'aught me language; and my profit' and upto[:-32] == 'You t' and same

37 characters up to "profit"; the window: 'aught me language; and my profit' ; dropped out: 'You t'
one learned position row per place: (32, 28)
the same 65 probabilities with or without the dropped characters: True


### ✋ Your turn

To predict what follows the whole line `FULL` (42 characters), which 32 characters does the model read? Put them in
`answer`. How many have dropped out?

In [15]:
answer = None   # TODO: the last 32 characters of FULL

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == FULL[-32:], 'the window is the last cfg.block_size characters: FULL[-32:]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [16]:
#@title Show a solution { display-mode: "form" }
answer = FULL[-cfg.block_size:]
print(repr(answer), "·", len(FULL) - len(answer), "characters dropped out:", repr(FULL[:-cfg.block_size]))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == FULL[-32:], 'the window is the last cfg.block_size characters: FULL[-32:]'
    print("✓ right:", answer)

" me language; and my profit on't" · 10 characters dropped out: 'You taught'
✓ right:  me language; and my profit on't


# Section 2 · Next character out

What comes out of the box, how one character is picked from it, and the loop that writes a passage.

## F0.4 · The probabilities: one forward pass, `[32]` → `[65]`

*What exactly comes out?*

**Predict first** (the page asks it too, before its clip): after our line, the 5 most probable characters take most of the total
of 1. What do the other 60 share between them: about 0.01, about 0.1 or about 0.3?

### F0.4.1 · Picture it

The next character will be exactly one of the characters in the model's vocabulary, so their probabilities split one whole,
1. Picture one bar, cut into a slice for each character: make one slice wider, and the others must shrink to make room. So
every slice is somewhere between nothing and the whole bar, and all the slices together are the bar. (Where the picture
breaks: in our model, no slice is ever exactly 0.)

### F0.4.2 · Watch it on our line

*(The page's recorded clip, in words.)*

32 ids go in, and the model runs once, from start to end. Out come 65 numbers, one per character of the vocabulary, in id
order. Each is a **probability**, a number from 0 to 1, and together they add up to 1. After our line `r` gets
0.340, `a` 0.210, `l` 0.133 and `o` 0.106; the other 60 split what is left,
0.106. One run, from 32 ids in to 65 probabilities out, is a **forward pass**.

### F0.4.3 · Read and run the code

One forward pass is the rest of `next_char_probs`. After a line that counts the ids, 3 lines are the inside of the box (M2 to
M5 open them): at each of the 32 places they give all 65 characters a score, 32 rows of 65 **logits**. The last line keeps the
last row, and **softmax** (M5) turns its 65 scores into probabilities. The cell runs the same steps, printing the shape after
each: they add up to 1, and even the smallest is above 0.

In [17]:
with torch.no_grad():
    x = model.tok_emb(ids) + model.pos_emb(torch.arange(len(ids)))       # a token row + a position row per place
    print('ids', tuple(ids.shape), '→ rows', tuple(x.shape))
    x = model.blocks(x.unsqueeze(0)).squeeze(0)                          # the block: attention, then feed-forward
    print('after the block', tuple(x.shape))
    logits = model.ln_f(x) @ model.tok_emb.weight.T + model.head_bias    # a score per character, at every place
    print('logits', tuple(logits.shape))
    probs = torch.softmax(logits[-1], dim=-1)                            # the last row → probabilities
    print('probabilities', tuple(probs.shape))
rest = 1 - probs.topk(5).values.sum().item()
print(f"they add up to {probs.sum().item():.6f}; the smallest is {probs.min().item():.1e}; the other 60 share {rest:.4f}")
assert tuple(logits.shape) == (32, 65) and tuple(probs.shape) == (65,) and torch.allclose(probs, p)
assert abs(probs.sum().item() - 1) < 1e-5 and probs.min() > 0 and abs(rest - 0.1063) < 5e-4

ids (32,) → rows (32, 28)
after the block (32, 28)
logits (32, 65)
probabilities (65,)
they add up to 1.000000; the smallest is 1.3e-13; the other 60 share 0.1063


### ✋ Your turn

How many of the 65 characters get a probability above 0.01 (one in a hundred) after our line? Put the count in `answer`.

In [18]:
answer = None   # TODO: count the entries of p above 0.01

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int((p > 0.01).sum()), '(p > 0.01) is True or False for each of the 65; .sum() counts the Trues'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [19]:
#@title Show a solution { display-mode: "form" }
answer = int((p > 0.01).sum())
print(answer, "characters:", [tok.chars[i] for i in torch.nonzero(p > 0.01).flatten().tolist()])

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int((p > 0.01).sum()), '(p > 0.01) is True or False for each of the 65; .sum() counts the Trues'
    print("✓ right:", answer)

7 characters: ['a', 'e', 'i', 'l', 'o', 'r', 'u']
✓ right: 7


## F0.5 · The pick: greedy or sampled

*How does it choose one character?*

**Predict first** (the page asks it too, before its clip): sampling draws one character by its probability. After our line, out
of 100 draws, about how many give `r`?

### F0.5.1 · Picture it

Picture a toy that writes whole words, always the most probable. Its table sends each word to just one next: *to*, *be*, *or*,
*not*, and *to* again. Start it at *to*: "to be or not to be or not", over and over. Same words in, same pick out.
Draw by probability instead, and now and then it takes a less likely word, such as *that*, and escapes. (Where the picture
breaks: our model picks characters, not words.)

### F0.5.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Writing needs exactly one character, not 65 probabilities. **Greedy decoding** always takes the most probable, here `r`. Run it
over and over and it writes *“rove the shall the shall the shall”*: round and round, greedy text loops. **Sampling** draws a character at random,
each as likely as its probability: `r` comes up about one time in three, `a` about one in five. One recorded sample from the
same line begins *“rown it, thes wont as but”*. Greedy is repeatable but loops; sampling varies and reads more like text.

### F0.5.3 · Read and run the code

Two functions in `whole_machine.py`, one for each way to pick. `write_greedy` appends `p.argmax()`, the id of the most probable
character; `write_sampled` appends `torch.multinomial(p, 1)`, one draw by probability, its draws fixed by a **seed**. Run the
three cells: greedy's text, 10,000 draws against the probabilities, and the seed-3 sample.

In [20]:
greedy = tok.decode(write_greedy(model, ids, 60))      # always the most probable, 60 times
print(repr(greedy))
assert greedy == 'rove the shall the shall the shall the shall the shall the s'

'rove the shall the shall the shall the shall the shall the s'


In [21]:
g = torch.Generator().manual_seed(0)
draws = torch.multinomial(p, 10_000, replacement=True, generator=g)   # 10,000 draws from the 65 probabilities
for c in 'ral':
    share = (draws == tok.stoi[c]).float().mean().item()
    print(f"{c!r}: drawn {share:.3f} of the time; its probability {p[tok.stoi[c]].item():.3f}")
    assert abs(share - p[tok.stoi[c]].item()) < 0.02

'r': drawn 0.343 of the time; its probability 0.340
'a': drawn 0.205 of the time; its probability 0.210
'l': drawn 0.132 of the time; its probability 0.133


In [22]:
sampled = tok.decode(write_sampled(model, ids, 240, seed=3))   # 240 draws, seed 3
print(sampled)
assert sampled == "rown it, thes wont as but thal'd;\nRyet. Why mie so in will like alther aftly.\n\nQUEESBEER:\nHow the Ment is trop then I memeliechpwats would,\nAnd the knees is fore my hearetle of a me't you,\nYo cutizen\ntratch the North my the thes this the do"   # the course's recorded sample from this line

rown it, thes wont as but thal'd;
Ryet. Why mie so in will like alther aftly.

QUEESBEER:
How the Ment is trop then I memeliechpwats would,
And the knees is fore my hearetle of a me't you,
Yo cutizen
tratch the North my the thes this the do


### ✋ Your turn

Run greedy twice, and sampling with seed 3 and with seed 4 (40 characters each is enough). Put two True or False values in
`answer`: `[greedy gave the same text both times, seeds 3 and 4 gave the same text]`.

In [23]:
answer = None   # TODO: [write_greedy(...) == write_greedy(...), write_sampled(..., seed=3) == write_sampled(..., seed=4)]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [True, False], 'greedy has no randomness; two different seeds make two different sets of draws'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [24]:
#@title Show a solution { display-mode: "form" }
answer = [write_greedy(model, ids, 40) == write_greedy(model, ids, 40),
          write_sampled(model, ids, 40, seed=3) == write_sampled(model, ids, 40, seed=4)]
print(repr(tok.decode(write_sampled(model, ids, 40, seed=4))))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == [True, False], 'greedy has no randomness; two different seeds make two different sets of draws'
    print("✓ right:", answer)

'oits?\n\nCAPULINA:\nAre whose ractself, the'
✓ right: [True, False]


## F0.6 · Generate: append, slide, run again

*How does it write a whole passage?*

**Predict first** (the page asks it too, before its clip): the model writes 240 characters from our line. How many of its 9,417
learned numbers change while it writes?

### F0.6.1 · Picture it

Each forward pass and its pick add one character, never a whole word, so every pick is a fork. After `th` the path may go to
`e` or to `a`: take `e` and the next run starts from `the`, take `a` and from `tha`. The texts grow apart. A pick is never
taken back, so one early draw shapes all that follows; our model's road splits 65 ways at every step.

### F0.6.2 · Watch it on our line

*(The page's recorded clip, in words.)*

The picked character is added to the end of the line. The window slides on by one, so it still holds 32 characters, and the
whole model runs again. Every new character costs one full forward pass: the recorded sample of 240
characters took 240 passes. Nothing inside the model changes while it writes: the 9,417 numbers stay
fixed, and only the window moves.

### F0.6.3 · Read and run the code

Writing is the loop inside `write_sampled`: each time round, `next_char_probs` runs the whole model on the text so far, its
window the last 32, and one draw is appended. The cell below is that loop written out, with a counter, and it checks that the
learned numbers are the same before and after: it copies all of them first, and compares.

In [25]:
before = {k: v.clone() for k, v in model.state_dict().items()}   # the learned numbers, before writing
g = torch.Generator().manual_seed(3)
text_ids, passes = tok.encode(LINE), 0
for step in range(240):
    window_ids = text_ids[-cfg.block_size:]                       # slide: the last 32 ids
    probs = next_char_probs(model, window_ids)                    # one forward pass: 65 probabilities
    passes += 1
    text_ids.append(int(torch.multinomial(probs, 1, generator=g)))   # pick one, and append it
    if step < 3:
        print(f"pass {passes}: reads {tok.decode(window_ids)!r} → {tok.chars[text_ids[-1]]!r}")
print(passes, 'forward passes for', len(text_ids) - len(LINE), 'new characters')
unchanged = all(torch.equal(before[k], v) for k, v in model.state_dict().items())
print('compared', sum(v.numel() for v in before.values()), 'learned numbers; unchanged:', unchanged)
assert tok.decode(text_ids[len(LINE):]) == sampled and passes == 240 and unchanged
assert sum(v.numel() for v in before.values()) == count_params(model)

pass 1: reads 'You taught me language; and my p' → 'r'
pass 2: reads 'ou taught me language; and my pr' → 'o'
pass 3: reads 'u taught me language; and my pro' → 'w'


240 forward passes for 240 new characters
compared 9417 learned numbers; unchanged: True


### ✋ Your turn

After the model has written its first 5 characters, which 32 characters does its window hold? Put them in `answer`
(as text).

In [26]:
answer = None   # TODO: the line plus the first 5 new characters, then the last 32

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.decode(text_ids[5:5 + 32]), 'add the 5 new characters to LINE, then keep the last 32: (LINE + sampled[:5])[-32:]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [27]:
#@title Show a solution { display-mode: "form" }
answer = (LINE + sampled[:5])[-cfg.block_size:]
print(repr(answer))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.decode(text_ids[5:5 + 32]), 'add the 5 new characters to LINE, then keep the last 32: (LINE + sampled[:5])[-32:]'
    print("✓ right:", answer)

'aught me language; and my prown '
✓ right: aught me language; and my prown 


**Next, M1:** the text the model learned from, and the one number its training makes small.